# 필수 패키지 불러오기 

In [1]:
# 필요한 페키지들을 불러옴
from pop import AI, Camera, Pilot, Util
import ipywidgets as wg
import cv2, time, numpy

In [2]:
# 카메라 인스턴스 생성
cam = Camera(300,300)

# 트랙 주행 모델 로드 

In [3]:
#트렉주행 모델을 불러옴
# Track_Follow 인스턴스 생성
TF=AI.Track_Drive(cam)
# Track_Follow 인스턴스 모델 로드
TF.load_model("Track_Model_S_v1.h5")     #성공
# Track_FOllow 인스턴스 run
TF.run()

{'x': 0.03292979, 'y': 0.07030235, 'cx': 9, 'cy': 10}

# 객체 탐지(Yolo) 모델 로드 

# 자동차 제어 객체 생성

In [4]:
# AutoCar 인스턴스 생성
Car=Pilot.AutoCar()
# 장애물이 있을 경우 정지하는 기능 disabled
Car.setObstacleDistance(0)

In [5]:
Car.steering = 0

# 트랙 주행 (신호 및 표지판 제외)

In [ ]:
# 초기 조향값, 속도 정의
steer = 0
speed = 40         

# 중앙 기준값 및 허용 오차 설정
CENTER_X = 0.0498               # 학습 기준 중앙 x값
ERROR_THRESHOLD = 0.028         # 너무 작으면 조향 민감도 커짐, 너무 크면 무시됨


# 비례 제어 계수 (조정 필요)
Kp = 50                         # 예: error * 50 → steer 범위 조정됨

#스피드:민감도:비례제어 / 25:0.025:50 / 30:0.028:50 / 40:0.033:48
#주행성공은 스피트 25

# 차량 출발
Car.forward(speed)

while True:
    try:
        # 이미지 캡처
        img = cam.value.copy()
        img1 = img.copy()

        # 모델 예측 실행
        t = TF.run(img1)

        # 에러 계산 (예측값 - 기준 중앙값)
        error = t['x'] - CENTER_X

        # 비례 제어 기반 조향값 계산
        if abs(error) < ERROR_THRESHOLD:
            steer = 0  # 너무 미세한 오차는 무시
        else:
            steer = error * Kp  # 비례 제어

        # 조향값 제한 (-1 ~ 1)
        steer = max(min(steer, 1), -1)

        # 차량에 조향 적용
        Car.steering = steer

        # 디버깅 출력
        print(f"x: {t['x']:.4f}, error: {error:.4f}, steer: {steer:.2f}")

        # 시각화: 조향 위치 표시
        img = cv2.circle(img, (int((steer + 1) / 2 * 300), 150), 6, (255, 0, 0), 2)

        # 이미지 출력 (Jupyter용)
        Util.imshow("img", img)

    except KeyboardInterrupt:
        # 인터럽트 시 차량 정지
        Car.stop()
        break


In [63]:
Car.stop()